# Exhibition interpolation — gh0st_flux_lora_v2

Assembles Screen A and Screen B exhibition videos from the FLUX-generated stills
produced by `generate_exhibition.py`.

**Screen A sequence:** stable → ambiguous → glitch → extreme → synthetic  
**Screen B sequence:** extreme → synthetic → stable → ambiguous → glitch

Within each category the stills run: pure → 30/70 mashup → 70/30 mashup → pure → …  
Category boundaries use the pre-generated boundary mashups.
Consecutive stills are bridged with Farneback optical flow.

**Before running:** upload the generated stills to Drive:
```
Local:  spikes/flux_lora_training/output/gh0st_exhibition_v1/
Drive:  Gh0st in the Loop/outputs/gh0st_exhibition_v1/
```
The folder structure should be:
```
gh0st_exhibition_v1/
  stable/      # pure stills + within-category mashups
  ambiguous/
  glitch/
  extreme/
  synthetic/
  boundaries/
    stable_x_ambiguous/
    ambiguous_x_glitch/
    …
```

In [ ]:
import os

!pip install opencv-python-headless imageio imageio-ffmpeg -q

from google.colab import drive
drive.mount('/content/drive')

if os.path.exists(repo_dir := '/content/gh0st-in-the-l00p'):
    !git -C {repo_dir} pull --quiet
else:
    !git clone --quiet https://github.com/jasonr2048/gh0st-in-the-l00p.git {repo_dir}

%cd {repo_dir}
print('Ready.')

In [ ]:
from datetime import datetime
from pathlib import Path
import re

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT    = Path('/content/drive/MyDrive/Gh0st in the Loop')
STILLS_BASE   = DRIVE_ROOT / 'outputs' / 'gh0st_exhibition_v1'
SELECTION     = Path('spikes/flux_lora_training/exhibition_source/selection_v1.txt')
OUTPUT_DIR    = DRIVE_ROOT / 'outputs'

# ── Screen sequences ──────────────────────────────────────────────────────────
SCREEN_A = ['stable', 'ambiguous', 'glitch', 'extreme', 'synthetic']
SCREEN_B = ['extreme', 'synthetic', 'stable', 'ambiguous', 'glitch']

# ── Timing ────────────────────────────────────────────────────────────────────
FPS          = 24
HOLD_FRAMES  = 0     # frames to hold each still before morphing
MORPH_FRAMES = 12    # optical flow frames between consecutive stills
                     # 12 @ 24fps = 0.5s per step

# ── Output size ───────────────────────────────────────────────────────────────
OUTPUT_SIZE  = (1024, 1024)   # (width, height) — keep square, crop in app.py

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_A = OUTPUT_DIR / f'exhibition_screen_A_{timestamp}.mp4'
OUT_B = OUTPUT_DIR / f'exhibition_screen_B_{timestamp}.mp4'
print(f'Screen A → {OUT_A.name}')
print(f'Screen B → {OUT_B.name}')

In [ ]:
# ── Parse selection file → {category: [source_name, ...]} ─────────────────────

def stem(name: str) -> str:
    """Same sanitisation as generate_exhibition.py."""
    s = Path(name).stem
    return re.sub(r'[\s()]+', '_', s).strip('_')

def parse_selection(path: Path) -> dict:
    categories = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = line.split('/', 1)
        if len(parts) != 2:
            continue
        cat, fname = parts
        categories.setdefault(cat, []).append(fname)
    return categories

selection = parse_selection(SELECTION)
for cat, files in selection.items():
    print(f'  {cat}: {len(files)} styles')

In [ ]:
# ── Build ordered still lists for each screen ─────────────────────────────────

def build_sequence(screen_order: list, selection: dict, base: Path) -> list:
    """Return ordered list of Paths for one screen."""
    stills = []
    for i, cat in enumerate(screen_order):
        files = selection[cat]
        stems = [stem(f) for f in files]
        cat_dir = base / cat

        # First pure still in category
        stills.append(cat_dir / f'{stems[0]}.png')

        # Within-category: mashup pair + next pure
        for j in range(len(stems) - 1):
            sa, sb = stems[j], stems[j + 1]
            stills.append(cat_dir / f'030{sa}__070{sb}.png')
            stills.append(cat_dir / f'070{sa}__030{sb}.png')
            stills.append(cat_dir / f'{sb}.png')

        # Boundary to next category
        if i < len(screen_order) - 1:
            next_cat = screen_order[i + 1]
            last_a  = stems[-1]
            first_b = stem(selection[next_cat][0])
            bdir = base / 'boundaries' / f'{cat}_x_{next_cat}'
            stills.append(bdir / f'030{last_a}__070{first_b}.png')
            stills.append(bdir / f'070{last_a}__030{first_b}.png')

    return stills


seq_a = build_sequence(SCREEN_A, selection, STILLS_BASE)
seq_b = build_sequence(SCREEN_B, selection, STILLS_BASE)

def report(name, seq):
    missing = [p for p in seq if not p.exists()]
    dur = ((len(seq) * HOLD_FRAMES) + (len(seq) - 1) * MORPH_FRAMES) / FPS
    print(f'{name}: {len(seq)} stills, {len(missing)} missing, ~{dur:.0f}s ({dur/60:.1f} min)')
    for m in missing:
        print(f'  MISSING: {m.relative_to(STILLS_BASE)}')

report('Screen A', seq_a)
report('Screen B', seq_b)

In [ ]:
# ── Preview: every 12th still from each screen ────────────────────────────────
from PIL import Image
import IPython.display as ipd

def thumb_strip(seq, step=12, label=''):
    sample = [p for p in seq[::step] if p.exists()]
    if not sample:
        print(f'{label}: no stills found yet')
        return
    thumbs = [Image.open(p).convert('RGB').resize((128, 128)) for p in sample]
    grid = Image.new('RGB', (len(thumbs) * 128, 128))
    for i, t in enumerate(thumbs):
        grid.paste(t, (i * 128, 0))
    print(label)
    ipd.display(grid)

thumb_strip(seq_a, label='Screen A (every 12th)')
thumb_strip(seq_b, label='Screen B (every 12th)')

In [ ]:
# ── Optical flow helpers ──────────────────────────────────────────────────────
import cv2
import numpy as np
from PIL import Image

def load(path: Path, size: tuple) -> np.ndarray:
    img = Image.open(path).convert('RGB')
    w, h = size
    src_w, src_h = img.size
    tgt_ratio = w / h
    src_ratio = src_w / src_h
    if abs(src_ratio - tgt_ratio) > 0.01:
        if src_ratio > tgt_ratio:
            new_w = int(src_h * tgt_ratio)
            left = (src_w - new_w) // 2
            img = img.crop((left, 0, left + new_w, src_h))
        else:
            new_h = int(src_w / tgt_ratio)
            top = (src_h - new_h) // 2
            img = img.crop((0, top, src_w, top + new_h))
    return np.array(img.resize((w, h), Image.LANCZOS))


def optical_flow_morph(a: np.ndarray, b: np.ndarray, steps: int) -> list:
    a_gray = cv2.cvtColor(a, cv2.COLOR_RGB2GRAY)
    b_gray = cv2.cvtColor(b, cv2.COLOR_RGB2GRAY)
    flow = cv2.calcOpticalFlowFarneback(
        a_gray, b_gray, None,
        pyr_scale=0.5, levels=3, winsize=15,
        iterations=3, poly_n=5, poly_sigma=1.2, flags=0
    )
    h, w = a.shape[:2]
    xs = np.tile(np.arange(w), (h, 1)).astype(np.float32)
    ys = np.tile(np.arange(h), (w, 1)).T.astype(np.float32)
    frames = []
    for t in np.linspace(0, 1, steps, endpoint=False):
        wx = (xs + flow[..., 0] * t).astype(np.float32)
        wy = (ys + flow[..., 1] * t).astype(np.float32)
        warped = cv2.remap(a, wx, wy, cv2.INTER_LINEAR)
        blended = cv2.addWeighted(warped, 1 - t, b, t, 0)
        frames.append(blended)
    return frames

print('Functions defined.')

In [ ]:
# ── Render function ───────────────────────────────────────────────────────────
import imageio

def render(seq: list, out_path: Path, label: str) -> float:
    """Render a sequence of stills to mp4 with optical flow. Returns duration."""
    existing = [p for p in seq if p.exists()]
    missing  = [p for p in seq if not p.exists()]
    if missing:
        print(f'WARNING: {len(missing)} stills missing — skipping them')
    if not existing:
        raise RuntimeError('No stills found — check STILLS_BASE path')

    print(f'Loading {len(existing)} stills...')
    frames_loaded = [load(p, OUTPUT_SIZE) for p in existing]
    print('Done loading.')

    all_frames = []
    for i, img in enumerate(frames_loaded):
        if HOLD_FRAMES > 0:
            all_frames.extend([img] * HOLD_FRAMES)
        if i < len(frames_loaded) - 1:
            all_frames.extend(optical_flow_morph(img, frames_loaded[i + 1], MORPH_FRAMES))
        if i % 20 == 0:
            print(f'  {i}/{len(frames_loaded)} stills processed ({len(all_frames)} frames)')

    if HOLD_FRAMES > 0:
        all_frames.extend([frames_loaded[-1]] * HOLD_FRAMES)

    total_s = len(all_frames) / FPS
    print(f'\nRendering {len(all_frames)} frames ({total_s:.1f}s) → {out_path.name}')
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with imageio.get_writer(str(out_path), fps=FPS) as writer:
        for i, frame in enumerate(all_frames):
            writer.append_data(frame)
            if i % 200 == 0:
                print(f'  {i}/{len(all_frames)} frames written...', end='\r')
    print(f'\n✅ {label} saved to {out_path}')
    return total_s

print('render() defined.')

In [ ]:
# ── Render Screen A ───────────────────────────────────────────────────────────
dur_a = render(seq_a, OUT_A, 'Screen A')

In [ ]:
# ── Render Screen B ───────────────────────────────────────────────────────────
dur_b = render(seq_b, OUT_B, 'Screen B')

In [ ]:
# ── Sidecar JSON ──────────────────────────────────────────────────────────────
import json

for out_path, dur, screen, seq in [
    (OUT_A, dur_a, 'A', seq_a),
    (OUT_B, dur_b, 'B', seq_b),
]:
    sidecar = {
        'experiment_id': out_path.stem,
        'screen': screen,
        'duration_seconds': round(dur, 3),
        'fps': FPS,
        'source': 'gh0st_exhibition_v1_flux_lora_v2',
        'n_stills': len(seq),
        'hold_frames': HOLD_FRAMES,
        'morph_frames': MORPH_FRAMES,
        'output_size': OUTPUT_SIZE,
        'generated_at': datetime.now().isoformat(),
    }
    sidecar_path = out_path.with_suffix('.json')
    sidecar_path.write_text(json.dumps(sidecar, indent=2))
    print(f'✅ Sidecar: {sidecar_path.name}')
    print(json.dumps(sidecar, indent=2))
    print()

In [ ]:
# ── In-notebook preview (Screen A) ───────────────────────────────────────────
from IPython.display import HTML
from base64 import b64encode

data_url = 'data:video/mp4;base64,' + b64encode(open(OUT_A, 'rb').read()).decode()
display(HTML(f'<video width=540 controls autoplay loop><source src="{data_url}" type="video/mp4"></video>'))